# AI-Based Skin Lesion Classification & Anatomical Site Identification (HAM10000)

**Educational / research prototype — not a clinically validated diagnostic system.**
It only distinguishes the 7 lesion categories in HAM10000 and will always output one of them,
even for an image of something else entirely. It must never be used to make or rule out a diagnosis.

| Task | Labels | Priority |
|---|---|---|
| **A. Lesion classification** | 7 diagnostic categories (`dx`) | Primary — target ≥ 80 % test accuracy |
| **B. Anatomical site** | `localization` metadata (back, face, trunk …) | Secondary — separate image classifier |

**Runtime:** `Runtime → Change runtime type → GPU` (T4 is fine). Run the cells top to bottom.
Checkpoints, models, plots and the final report are saved to Google Drive, so a disconnect does not lose finished work
(re-running a training cell with `RESUME = True` loads the finished model instead of retraining).

**Expected run time on a T4:** ≈ 1.5–3 h for everything (architecture comparison is the largest part;
set `RUN_ARCH_COMPARISON = False` to train only `DEFAULT_ARCH`).

## 0. Execution strategy (analysis before implementation)

**Dataset structure.** 10,015 dermatoscopic JPEGs (600×450) from ~7,470 unique lesions. `HAM10000_metadata.csv` has
`lesion_id, image_id, dx, dx_type, age, sex, localization`. Several images often show **the same lesion** (same
`lesion_id`), so a naive random split leaks near-identical images into the test set and inflates accuracy.

**Plan.**
1. Download with the Kaggle API, verify every image file exists and decodes; drop corrupted ones.
2. EDA: missing values, duplicate images (same lesion and byte-identical files), class and site distribution.
3. **Split at lesion level** (all images of a lesion go to one split), stratified by `dx`: 70 / 15 / 15.
   The test set is not touched until the final evaluation section.
4. Transfer learning with ImageNet weights; 3-stage gradual fine-tuning (head → top 30 % → whole network,
   BatchNorm kept frozen), augmentation, dropout + L2, early stopping, LR reduction on plateau, checkpointing.
5. Compare EfficientNetB0, EfficientNetB3, MobileNetV2, ResNet50 with an identical short schedule and **choose by
   validation accuracy** (macro-F1 as tie-break). Retrain the winner with the full schedule.
6. Decide on test-time augmentation (TTA) **using validation only**, then evaluate once on test.
7. Train a separate site classifier on the same lesion-level split; compare against a majority-class baseline.
8. Grad-CAM, upload-and-predict interface, saved models, auto-generated performance report.

**Risks to the 80 % target and how they are handled.**

| Risk | Effect | Mitigation |
|---|---|---|
| Heavy imbalance (`nv` ≈ 67 %) | A model predicting only `nv` already scores ≈ 67 %; accuracy alone is misleading | Report macro-F1, balanced accuracy and per-class recall next to accuracy; mild (√) class weighting |
| Class weighting vs. accuracy | Strong weighting raises minority recall but usually *lowers* overall accuracy | `CLASS_WEIGHT_POWER` knob: 0 = none, 0.5 = √ (default compromise), 1 = fully balanced |
| Lesion-level split | Lower (honest) accuracy than random image splits seen in many notebooks | Kept on purpose; we also report one-image-per-lesion test accuracy |
| `mel` vs `nv` vs `bkl` confusion | Main source of errors in the literature | Higher resolution (B3 @ 300 px), longer fine-tuning, TTA |
| Small minority classes (`df` ≈ 115 imgs, `vasc` ≈ 142) | Noisy per-class metrics | Reported with support counts; stratified split keeps them in every split |
| Site labels barely visible in close-up dermoscopy | Site accuracy will be far below lesion accuracy | Treated as a secondary task; judged against majority baseline, not against 80 % |

**Why TensorFlow/Keras:** all four required backbones ship with ImageNet weights in `keras.applications`,
Grad-CAM is straightforward, and the `.keras` format is easy to save, download and reload.

## 1. Setup — runtime, imports, configuration

In [ ]:
# GPU check
!nvidia-smi -L || echo "No GPU detected - switch Runtime > Change runtime type > GPU"
!pip -q install -U kaggle   # recent CLI supports the new KAGGLE_API_TOKEN

In [ ]:
import os, gc, json, glob, time, random, hashlib, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, top_k_accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
GPUS = tf.config.list_physical_devices("GPU")
print("GPUs:", GPUS)
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# ============================ CONFIGURATION ============================
SEED = 42

# Where results live. On Colab this is on Google Drive so nothing is lost on disconnect.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/skin_lesion_project"
LOCAL_PROJECT_DIR = "./skin_lesion_project"          # used when not on Colab
DATA_DIR = "/content/ham10000" if IN_COLAB else "./ham10000"
KAGGLE_DATASET = "kmader/skin-cancer-mnist-ham10000"

RESUME = True                     # load an already-finished model from Drive instead of retraining
RUN_ARCH_COMPARISON = True        # train all 4 backbones on a short schedule and pick by validation
DEFAULT_ARCH = "EfficientNetB3"   # used if RUN_ARCH_COMPARISON = False
ARCHS = ["EfficientNetB0", "EfficientNetB3", "MobileNetV2", "ResNet50"]

# Image size per architecture (EfficientNetB3's native resolution is 300)
IMG_SIZES = {"EfficientNetB0": 224, "EfficientNetB3": 300, "MobileNetV2": 224, "ResNet50": 224}
BATCH_SIZE = 32
PRETRAINED_WEIGHTS = "imagenet"
USE_MIXED_PRECISION = bool(GPUS)  # faster on T4; outputs are kept in float32

# Gradual fine-tuning schedule: list of (fraction of backbone unfrozen, max epochs, learning rate)
COMPARE_SCHEDULE = [(0.0, 2, 1e-3), (0.3, 4, 1e-4), (1.0, 4, 3e-5)]   # short, identical for all archs
FINAL_SCHEDULE   = [(0.0, 3, 1e-3), (0.3, 10, 1e-4), (1.0, 20, 2e-5)] # full schedule for the chosen arch
LOC_SCHEDULE     = [(0.0, 3, 1e-3), (0.3, 8, 1e-4), (1.0, 10, 2e-5)]
EARLY_STOP_PATIENCE = 4
LR_PLATEAU_PATIENCE = 2

# Imbalance handling: per-class weight = (N / (K * n_c)) ** CLASS_WEIGHT_POWER
#   0.0 -> no weighting (usually best raw accuracy), 0.5 -> sqrt (compromise), 1.0 -> fully balanced
CLASS_WEIGHT_POWER = 0.5
LOSS = "cce"                      # "cce" (with label smoothing) or "focal"
LABEL_SMOOTHING = 0.1
DROPOUT = 0.4
L2 = 1e-4

# Anatomical-site task
LOC_ARCH = None                   # None -> same backbone as the chosen disease model
LOC_MIN_IMAGES = 40               # site classes with fewer images are excluded (e.g. 'acral' ~7 images)

TARGET_ACC = 0.80
LOW_CONFIDENCE_THRESHOLD = 0.50   # interface warns when top-1 probability is below this
# ======================================================================

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything()

if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision:", USE_MIXED_PRECISION)

## 2. Google Drive (persistent storage for checkpoints, models and reports)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
else:
    PROJECT_DIR = Path(LOCAL_PROJECT_DIR)

DIRS = {k: PROJECT_DIR / k for k in ["checkpoints", "models", "figures", "histories", "reports"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("Project dir:", PROJECT_DIR.resolve())

def savefig(name):
    path = DIRS["figures"] / f"{name}.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    return path

## 3. Download the dataset with the Kaggle API

Provide credentials in **one** of two ways:
* **Colab Secrets (recommended):** in the 🔑 panel add either `KAGGLE_API_TOKEN` (the token Kaggle → Settings → API shows) or the legacy pair `KAGGLE_USERNAME` + `KAGGLE_KEY`, and switch **Notebook access** on, or
* **Upload `kaggle.json`** (Kaggle → Settings → API → *Create New Token*) when prompted.

The cell skips the download if the images are already present.

In [ ]:
def have_images(root):
    return len(glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True)) >= 10000

def setup_kaggle_credentials():
    # Accepts either Kaggle's newer single API token (KAGGLE_API_TOKEN) or the legacy username + key / kaggle.json
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return
    if IN_COLAB:
        from google.colab import userdata
        def secret(name):
            try:
                return userdata.get(name)
            except Exception:
                return None
        if secret("KAGGLE_API_TOKEN"):
            os.environ["KAGGLE_API_TOKEN"] = secret("KAGGLE_API_TOKEN")
            print("Using KAGGLE_API_TOKEN from Colab Secrets."); return
        if secret("KAGGLE_USERNAME") and secret("KAGGLE_KEY"):
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
            print("Using KAGGLE_USERNAME / KAGGLE_KEY from Colab Secrets."); return
        print("No Kaggle secrets found (or notebook access not enabled for them).")
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        if not IN_COLAB:
            raise RuntimeError("Place kaggle.json in ~/.kaggle/ or set KAGGLE_USERNAME / KAGGLE_KEY.")
        from google.colab import files
        print("Upload your kaggle.json")
        up = files.upload()
        kaggle_json.parent.mkdir(exist_ok=True)
        kaggle_json.write_bytes(up[next(iter(up))])
    os.chmod(kaggle_json, 0o600)

if have_images(DATA_DIR):
    print("Dataset already present in", DATA_DIR)
else:
    setup_kaggle_credentials()
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip -q
    assert have_images(DATA_DIR), "Download/extraction failed - check credentials and disk space."
print(sorted(os.listdir(DATA_DIR)))

## 4. Load metadata, check paths, missing values, corrupted files and duplicates

In [ ]:
meta_files = glob.glob(os.path.join(DATA_DIR, "**", "HAM10000_metadata.csv"), recursive=True)
assert meta_files, f"HAM10000_metadata.csv not found under {DATA_DIR}"
df = pd.read_csv(meta_files[0])
print("Metadata rows:", len(df))

# The Kaggle archive contains the images twice (HAM10000_images_part_1 and ham10000_images_part_1).
# Map image_id -> one path; duplicates of the same file collapse naturally.
path_map = {}
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True)):
    path_map.setdefault(Path(p).stem, p)
df["path"] = df["image_id"].map(path_map)

missing_files = df["path"].isna().sum()
print("Images listed in metadata but not found on disk:", missing_files)
df = df.dropna(subset=["path"]).reset_index(drop=True)
display(df.head())

In [ ]:
print("Missing values per column:")
print(df.isna().sum().to_string())
print("\n'unknown' values:", {c: int((df[c] == "unknown").sum()) for c in ["sex", "localization"]})

# Each lesion must carry exactly one diagnosis, otherwise lesion-level splitting would be ambiguous
dx_per_lesion = df.groupby("lesion_id")["dx"].nunique()
assert (dx_per_lesion == 1).all(), "Some lesions have multiple diagnoses"
imgs_per_lesion = df.groupby("lesion_id").size()
print(f"\nImages: {len(df)} | unique lesions: {df['lesion_id'].nunique()} | "
      f"lesions with >1 image: {(imgs_per_lesion > 1).sum()} (max {imgs_per_lesion.max()} images)")
print("Duplicate image_id rows:", df["image_id"].duplicated().sum())

In [ ]:
# Verify every image decodes and find byte-identical files (exact duplicates)
def check_image(path):
    try:
        with open(path, "rb") as f:
            data = f.read()
        with Image.open(path) as im:
            im.verify()
        return hashlib.md5(data).hexdigest(), True
    except Exception:
        return None, False

from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    results = list(ex.map(check_image, df["path"]))
df["md5"] = [r[0] for r in results]
df["readable"] = [r[1] for r in results]
print(f"Checked {len(df)} files in {time.time()-t0:.0f}s")

bad = df[~df["readable"]]
print("Corrupted/unreadable images:", len(bad))
df = df[df["readable"]].reset_index(drop=True)

dup = df[df["md5"].duplicated(keep=False)].sort_values("md5")
print("Byte-identical image files:", len(dup))
if len(dup):
    cross_lesion = dup.groupby("md5")["lesion_id"].nunique()
    print("  ...of which shared across different lesion_ids:", int((cross_lesion > 1).sum()))
    # Keep one copy of each byte-identical file so the same pixels cannot sit in two splits
    df = df.drop_duplicates("md5").reset_index(drop=True)
print("Images after cleaning:", len(df))

## 5. Exploratory data analysis

In [ ]:
DX_NAMES = {
    "nv": "Melanocytic nevi", "mel": "Melanoma", "bkl": "Benign keratosis-like lesions",
    "bcc": "Basal cell carcinoma", "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "vasc": "Vascular lesions", "df": "Dermatofibroma",
}
DX_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]   # fixed order = model output order

counts = df["dx"].value_counts().reindex(DX_CLASSES)
lesion_counts = df.drop_duplicates("lesion_id")["dx"].value_counts().reindex(DX_CLASSES)
dist = pd.DataFrame({"name": [DX_NAMES[c] for c in DX_CLASSES], "images": counts.values,
                     "% images": (100 * counts / counts.sum()).round(1).values,
                     "unique lesions": lesion_counts.values}, index=DX_CLASSES)
display(dist)
print(f"Majority-class baseline (always 'nv'): {counts['nv']/counts.sum():.1%} accuracy  "
      f"<- the 80% target must be read against this")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
counts.sort_values().plot.barh(ax=ax[0], color="steelblue"); ax[0].set_title("Images per diagnosis")
df["dx_type"].value_counts().plot.bar(ax=ax[1], color="gray"); ax[1].set_title("How the diagnosis was confirmed (dx_type)")
plt.tight_layout(); savefig("eda_class_distribution"); plt.show()

In [ ]:
# Sample images from all seven categories
fig, axes = plt.subplots(len(DX_CLASSES), 5, figsize=(13, 2.3 * len(DX_CLASSES)))
for i, c in enumerate(DX_CLASSES):
    sample = df[df["dx"] == c].sample(min(5, (df["dx"] == c).sum()), random_state=SEED)
    for j in range(5):
        ax = axes[i, j]; ax.axis("off")
        if j < len(sample):
            ax.imshow(Image.open(sample.iloc[j]["path"]))
        if j == 0:
            ax.set_title(f"{c} — {DX_NAMES[c]}", loc="left", fontsize=10)
plt.tight_layout(); savefig("eda_samples"); plt.show()

In [ ]:
# Anatomical site distribution and relation to diagnosis
loc_counts = df["localization"].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
loc_counts.sort_values().plot.barh(ax=ax[0], color="darkseagreen"); ax[0].set_title("Images per anatomical site")
ct = pd.crosstab(df["localization"], df["dx"], normalize="index")[DX_CLASSES]
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues", ax=ax[1], cbar=False)
ax[1].set_title("Diagnosis mix within each site (row-normalised)")
plt.tight_layout(); savefig("eda_localization"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
df["age"].plot.hist(bins=20, ax=ax[0], color="slateblue"); ax[0].set_title("Age")
df["sex"].value_counts().plot.bar(ax=ax[1], color="slateblue"); ax[1].set_title("Sex")
plt.tight_layout(); plt.show()

## 6. Lesion-level stratified split (70 / 15 / 15)

Splitting is done on **unique lesions**, stratified by diagnosis; every image of a lesion inherits its lesion's split.
The assertions below prove no lesion, image or identical file appears in two splits.

In [ ]:
lesions = df.drop_duplicates("lesion_id")[["lesion_id", "dx"]]
train_les, temp_les = train_test_split(lesions, test_size=0.30, stratify=lesions["dx"], random_state=SEED)
val_les, test_les = train_test_split(temp_les, test_size=0.50, stratify=temp_les["dx"], random_state=SEED)

split_of = {**{l: "train" for l in train_les["lesion_id"]},
            **{l: "val" for l in val_les["lesion_id"]},
            **{l: "test" for l in test_les["lesion_id"]}}
df["split"] = df["lesion_id"].map(split_of)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

# ---- leakage checks ----
for a, b, na, nb in [(train_df, val_df, "train", "val"), (train_df, test_df, "train", "test"), (val_df, test_df, "val", "test")]:
    for col in ["lesion_id", "image_id", "md5"]:
        overlap = set(a[col]) & set(b[col])
        assert not overlap, f"Leakage: {len(overlap)} shared {col} between {na} and {nb}"
print("No lesion / image / file overlap between splits ✔")

split_table = pd.DataFrame({s: d["dx"].value_counts().reindex(DX_CLASSES) for s, d in
                            [("train", train_df), ("val", val_df), ("test", test_df)]})
split_table.loc["TOTAL"] = split_table.sum()
split_table["test %"] = (100 * split_table["test"] / split_table[["train", "val", "test"]].sum(axis=1)).round(1)
display(split_table)
print({s: f"{len(d)} images ({len(d)/len(df):.1%})" for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]})
df[["image_id", "lesion_id", "dx", "localization", "split"]].to_csv(PROJECT_DIR / "split.csv", index=False)

## 7. Class-imbalance handling

Per-class sample weights `w_c = (N / (K·n_c))^p`, normalised to mean 1 over the training set.
`p = CLASS_WEIGHT_POWER` (default 0.5). Full balancing (`p = 1`) maximises minority recall but costs overall accuracy;
`p = 0` maximises accuracy but neglects melanoma and other minority classes. The compromise is deliberate and configurable.

In [ ]:
def class_weights(labels, classes, power=CLASS_WEIGHT_POWER):
    counts = pd.Series(labels).value_counts().reindex(classes).fillna(0).values.astype(float)
    w = (counts.sum() / (len(classes) * np.maximum(counts, 1))) ** power
    w = w / (w * counts).sum() * counts.sum()      # mean sample weight = 1
    return dict(zip(classes, w))

DX_WEIGHTS = class_weights(train_df["dx"], DX_CLASSES)
display(pd.DataFrame({"train images": train_df["dx"].value_counts().reindex(DX_CLASSES),
                      "weight": pd.Series(DX_WEIGHTS).round(3)}))

## 8. Preprocessing and augmentation pipeline (`tf.data`)

* Decode → resize to the backbone's resolution → cache (uint8, in RAM).
* Training-only augmentation: flips, 90° rotations, random zoom-crop, brightness / contrast / saturation / hue jitter
  (dermoscopy images have no canonical orientation, so rotations and flips are label-preserving).
* Backbone-specific normalisation is applied **outside** the model and stored in the model config, so inference uses exactly the same preprocessing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_for(arch):
    # Maps float32 RGB images in [0, 255] to what each ImageNet backbone expects.
    if arch.startswith("EfficientNet"):
        return lambda x: x                                    # EfficientNet normalises internally
    if arch == "MobileNetV2":
        return lambda x: x / 127.5 - 1.0                      # [-1, 1]
    if arch == "ResNet50":
        mean = tf.constant([103.939, 116.779, 123.68])
        return lambda x: x[..., ::-1] - mean                  # 'caffe': RGB->BGR, mean subtraction
    raise ValueError(arch)

def load_and_resize(path, size):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (size, size), antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def augment(img, size):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    scale = tf.random.uniform([], 0.80, 1.0)                  # random zoom (crop 80-100% then resize back)
    crop = tf.cast(scale * size, tf.int32)
    img = tf.image.random_crop(img, (crop, crop, 3))
    img = tf.image.resize(img, (size, size))
    img = tf.image.random_brightness(img, 0.1 * 255)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    img = tf.image.random_hue(img, 0.03)                      # small: colour carries diagnostic signal
    return tf.clip_by_value(img, 0.0, 255.0)

def make_dataset(frame, label_col, classes, arch, training=False, weights=None, cache=True):
    size = IMG_SIZES[arch]
    prep = preprocess_for(arch)
    idx = {c: i for i, c in enumerate(classes)}
    y = frame[label_col].map(idx).values.astype("int32")
    w = (frame[label_col].map(weights).values if weights else np.ones(len(frame))).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, y, w))
    ds = ds.map(lambda p, y, w: (load_and_resize(p, size), y, w), num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def finish(img, y, w):
        img = tf.cast(img, tf.float32)
        if training:
            img = augment(img, size)
        out = (prep(img), tf.one_hot(y, len(classes)))
        return out + (w,) if training else out
    ds = ds.map(finish, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

# Preview augmentation (uses an uncached pipeline)
size = 224
ex = train_df.sample(4, random_state=1)["path"].values
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for i, p in enumerate(ex):
    raw = tf.cast(load_and_resize(p, size), tf.float32)
    axes[i, 0].imshow(raw.numpy().astype("uint8")); axes[i, 0].set_title("original")
    for j in range(1, 6):
        axes[i, j].imshow(augment(raw, size).numpy().astype("uint8")); axes[i, j].set_title("augmented")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 9. Model builder and the gradual fine-tuning trainer

* Backbone (ImageNet) → GlobalAveragePooling → Dropout → Dense logits (L2) → softmax. The logits layer is kept separate for Grad-CAM.
* Stage 1 trains only the head; stage 2 unfreezes the top 30 % of the backbone; stage 3 unfreezes all of it at a small LR.
  BatchNorm layers stay frozen throughout (standard practice for small-batch fine-tuning).
* Each stage: early stopping, `ReduceLROnPlateau`, a checkpoint on Drive, and the best-validation weights restored at the end of the stage.

In [ ]:
GRADCAM_LAYERS = {"EfficientNetB0": "top_activation", "EfficientNetB3": "top_activation",
                  "MobileNetV2": "out_relu", "ResNet50": "conv5_block3_out"}

def build_model(arch, num_classes, name):
    size = IMG_SIZES[arch]
    inputs = keras.Input((size, size, 3), name="image")
    ctor = getattr(keras.applications, arch)
    # input_tensor= keeps the backbone layers in the outer graph, so Grad-CAM can reach the last conv layer directly
    base = ctor(include_top=False, weights=PRETRAINED_WEIGHTS, input_tensor=inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    logits = layers.Dense(num_classes, kernel_regularizer=keras.regularizers.l2(L2),
                          dtype="float32", name="logits")(x)
    probs = layers.Activation("softmax", dtype="float32", name="probs")(logits)
    return keras.Model(inputs, probs, name=name), base

def set_trainable(base, fraction):
    # Unfreeze the top `fraction` of backbone layers; BatchNorm always stays frozen.
    base.trainable = True
    cutoff = int(len(base.layers) * (1 - fraction))
    for i, l in enumerate(base.layers):
        l.trainable = (i >= cutoff) and not isinstance(l, layers.BatchNormalization)

def make_loss():
    if LOSS == "focal":
        return keras.losses.CategoricalFocalCrossentropy(gamma=2.0, label_smoothing=LABEL_SMOOTHING)
    return keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

class BestWeights(keras.callbacks.Callback):
    # Tracks the best val_accuracy across a stage and restores those weights at the end of the stage.
    def __init__(self):
        super().__init__(); self.best = -np.inf; self.weights = None
    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_accuracy", -np.inf)
        if v > self.best:
            self.best, self.weights = v, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.weights is not None:
            self.model.set_weights(self.weights)

def save_inference_model(model, path):
    # Rebuild from config so the saved file has no optimizer state (~3x smaller; identical predictions)
    clean = model.__class__.from_config(model.get_config())
    clean.set_weights(model.get_weights())
    clean.save(path)

def train_model(arch, classes, train_frame, val_frame, label_col, weights, schedule, run_name):
    final_path = DIRS["models"] / f"{run_name}.keras"
    hist_path = DIRS["histories"] / f"{run_name}.json"
    if RESUME and final_path.exists() and hist_path.exists():
        print(f"[{run_name}] found finished model on Drive - loading instead of training")
        return keras.models.load_model(final_path), json.loads(hist_path.read_text())

    seed_everything()
    keras.backend.clear_session()
    model, base = build_model(arch, len(classes), run_name)
    train_ds = make_dataset(train_frame, label_col, classes, arch, training=True, weights=weights)
    val_ds = make_dataset(val_frame, label_col, classes, arch)

    history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": [], "stage_ends": []}
    best_val, best_weights = -np.inf, None
    t0 = time.time()
    for s, (fraction, epochs, lr) in enumerate(schedule, start=1):
        if fraction == 0:
            base.trainable = False
        else:
            set_trainable(base, fraction)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss=make_loss(),
                      metrics=["accuracy"])
        n_train = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
        print(f"\n[{run_name}] stage {s}: unfreeze {fraction:.0%} of backbone, lr={lr:g}, "
              f"trainable params={n_train:,}")
        bw = BestWeights()
        cbs = [bw,
               keras.callbacks.EarlyStopping("val_accuracy", mode="max", patience=EARLY_STOP_PATIENCE),
               keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.3, patience=LR_PLATEAU_PATIENCE, min_lr=1e-7),
               keras.callbacks.ModelCheckpoint(str(DIRS["checkpoints"] / f"{run_name}_stage{s}.keras"),
                                               monitor="val_accuracy", mode="max", save_best_only=True)]
        h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=2)
        for k in ["accuracy", "val_accuracy", "loss", "val_loss"]:
            history[k] += [float(v) for v in h.history[k]]
        history["stage_ends"].append(len(history["accuracy"]))
        if bw.best > best_val:
            best_val, best_weights = bw.best, bw.weights
        print(f"[{run_name}] stage {s} best val_accuracy = {bw.best:.4f}")

    model.set_weights(best_weights)          # best validation weights over all stages
    history["best_val_accuracy"] = float(best_val)
    history["train_minutes"] = (time.time() - t0) / 60
    save_inference_model(model, final_path)
    hist_path.write_text(json.dumps(history))
    print(f"[{run_name}] done in {history['train_minutes']:.1f} min, best val_accuracy {best_val:.4f} -> {final_path}")
    del train_ds, val_ds; gc.collect()
    return model, history

In [ ]:
def predict_probs(model, frame, label_col, classes, arch, tta=False):
    ds = make_dataset(frame, label_col, classes, arch, cache=False)
    probs, n = 0, 0
    views = [lambda x: x]
    if tta:   # flips commute with per-pixel normalisation, so they can be applied after preprocessing
        views += [lambda x: tf.image.flip_left_right(x), lambda x: tf.image.flip_up_down(x),
                  lambda x: tf.image.rot90(x, k=1)]
    for v in views:
        probs = probs + model.predict(ds.map(lambda x, y: v(x)), verbose=0); n += 1
    return probs / n

def metrics_from_probs(y_true, probs, classes):
    y_pred = probs.argmax(1)
    labels = list(range(len(classes)))
    out = {"accuracy": accuracy_score(y_true, y_pred),
           "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
           "macro_f1": f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
           "weighted_f1": f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0),
           "macro_precision": precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
           "macro_recall": recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)}
    if len(classes) > 2:
        out["top3_accuracy"] = top_k_accuracy_score(y_true, probs, k=3, labels=labels)
    try:
        present = np.unique(y_true)
        p = probs[:, present] / probs[:, present].sum(1, keepdims=True)
        out["macro_roc_auc_ovr"] = roc_auc_score(np.searchsorted(present, y_true), p, multi_class="ovr", average="macro") \
            if len(present) > 2 else np.nan
    except ValueError:
        out["macro_roc_auc_ovr"] = np.nan
    return out

def y_index(frame, label_col, classes):
    idx = {c: i for i, c in enumerate(classes)}
    return frame[label_col].map(idx).values

## 10. Architecture comparison (selection on the validation set only)

All four backbones get the identical `COMPARE_SCHEDULE`. The winner is the highest validation accuracy
(macro-F1 breaks near-ties within 0.5 pp). The test set is not used here.

In [ ]:
comparison = []
if RUN_ARCH_COMPARISON:
    y_val = y_index(val_df, "dx", DX_CLASSES)
    for arch in ARCHS:
        model, hist = train_model(arch, DX_CLASSES, train_df, val_df, "dx", DX_WEIGHTS,
                                  COMPARE_SCHEDULE, run_name=f"compare_{arch}")
        m = metrics_from_probs(y_val, predict_probs(model, val_df, "dx", DX_CLASSES, arch), DX_CLASSES)
        comparison.append({"architecture": arch, "img_size": IMG_SIZES[arch],
                           "params_M": round(model.count_params() / 1e6, 1),
                           "train_min": round(hist.get("train_minutes", float("nan")), 1),
                           **{f"val_{k}": v for k, v in m.items()}})
        del model; keras.backend.clear_session(); gc.collect()
    comp_df = pd.DataFrame(comparison).set_index("architecture")
    top = comp_df["val_accuracy"].max()
    contenders = comp_df[comp_df["val_accuracy"] >= top - 0.005]
    BEST_ARCH = contenders["val_macro_f1"].idxmax()
    display(comp_df.round(4))
    print("Selected architecture (validation):", BEST_ARCH)
else:
    comp_df = pd.DataFrame()
    BEST_ARCH = DEFAULT_ARCH
    print("Skipping comparison; using", BEST_ARCH)

## 11. Train the final disease classifier (chosen architecture, full schedule)

In [ ]:
disease_model, disease_hist = train_model(BEST_ARCH, DX_CLASSES, train_df, val_df, "dx", DX_WEIGHTS,
                                          FINAL_SCHEDULE, run_name=f"disease_{BEST_ARCH}")

In [ ]:
def plot_history(hist, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ep = np.arange(1, len(hist["accuracy"]) + 1)
    for a, key, lab in [(ax[0], "accuracy", "Accuracy"), (ax[1], "loss", "Loss")]:
        a.plot(ep, hist[key], label=f"train {key}")
        a.plot(ep, hist[f"val_{key}"], label=f"val {key}")
        for e in hist.get("stage_ends", [])[:-1]:
            a.axvline(e + 0.5, color="gray", ls="--", lw=0.8)
        a.set_xlabel("epoch (dashed = next fine-tuning stage)"); a.set_title(f"{title} — {lab}"); a.legend()
    plt.tight_layout(); savefig(fname); plt.show()

plot_history(disease_hist, f"Disease model ({BEST_ARCH})", "disease_training_curves")

## 12. Test-time augmentation decision (validation only)

TTA averages predictions over the original image, horizontal flip, vertical flip and a 90° rotation.
It is enabled for the test evaluation only if it improves **validation** accuracy.

In [ ]:
y_val = y_index(val_df, "dx", DX_CLASSES)
val_plain = metrics_from_probs(y_val, predict_probs(disease_model, val_df, "dx", DX_CLASSES, BEST_ARCH), DX_CLASSES)
val_tta = metrics_from_probs(y_val, predict_probs(disease_model, val_df, "dx", DX_CLASSES, BEST_ARCH, tta=True), DX_CLASSES)
USE_TTA = val_tta["accuracy"] > val_plain["accuracy"]
display(pd.DataFrame({"no TTA": val_plain, "TTA": val_tta}).round(4))
print("Use TTA on test:", USE_TTA)
VAL_METRICS = val_tta if USE_TTA else val_plain

## 13. Final evaluation on the held-out test set

Run once. Everything above was decided without looking at these images.

In [ ]:
y_test = y_index(test_df, "dx", DX_CLASSES)
test_probs = predict_probs(disease_model, test_df, "dx", DX_CLASSES, BEST_ARCH, tta=USE_TTA)
TEST_METRICS = metrics_from_probs(y_test, test_probs, DX_CLASSES)
test_pred = test_probs.argmax(1)

# Stricter view: one image per lesion (removes the advantage of lesions photographed many times)
one_per_lesion = test_df.groupby("lesion_id").head(1).index.values
TEST_METRICS_1PL = metrics_from_probs(y_test[one_per_lesion], test_probs[one_per_lesion], DX_CLASSES)

summary = pd.DataFrame({"validation": VAL_METRICS, "TEST (all images)": TEST_METRICS,
                        "TEST (1 image / lesion)": TEST_METRICS_1PL}).round(4)
display(summary)
status = "MET ✔" if TEST_METRICS["accuracy"] >= TARGET_ACC else "NOT met ✘ - see section 14"
print(f"\nTest accuracy {TEST_METRICS['accuracy']:.2%} | target {TARGET_ACC:.0%}: {status}")
print(f"Majority-class baseline on this test set: {(y_test == DX_CLASSES.index('nv')).mean():.2%}")

In [ ]:
print(classification_report(y_test, test_pred, labels=range(len(DX_CLASSES)),
                            target_names=[f"{c} ({DX_NAMES[c]})" for c in DX_CLASSES], digits=3, zero_division=0))

cm = confusion_matrix(y_test, test_pred, labels=range(len(DX_CLASSES)))
per_class = pd.DataFrame({
    "support": cm.sum(1),
    "per-class accuracy (recall)": np.diag(cm) / np.maximum(cm.sum(1), 1),
    "precision": precision_score(y_test, test_pred, labels=range(len(DX_CLASSES)), average=None, zero_division=0),
    "f1": f1_score(y_test, test_pred, labels=range(len(DX_CLASSES)), average=None, zero_division=0),
}, index=DX_CLASSES)
aucs = []
for i in range(len(DX_CLASSES)):
    yb = (y_test == i).astype(int)
    aucs.append(roc_auc_score(yb, test_probs[:, i]) if 0 < yb.sum() < len(yb) else np.nan)
per_class["ROC-AUC (one-vs-rest)"] = aucs
PER_CLASS_TEST = per_class
display(per_class.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=DX_CLASSES, yticklabels=DX_CLASSES, ax=ax[0])
ax[0].set(title="Confusion matrix (counts)", xlabel="Predicted", ylabel="True")
sns.heatmap(cm / np.maximum(cm.sum(1, keepdims=True), 1), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=DX_CLASSES, yticklabels=DX_CLASSES, ax=ax[1])
ax[1].set(title="Confusion matrix (row-normalised = recall)", xlabel="Predicted", ylabel="True")
plt.tight_layout(); savefig("disease_confusion_matrix"); plt.show()

plt.figure(figsize=(7, 6))
for i, c in enumerate(DX_CLASSES):
    yb = (y_test == i).astype(int)
    if 0 < yb.sum() < len(yb):
        fpr, tpr, _ = roc_curve(yb, test_probs[:, i])
        plt.plot(fpr, tpr, label=f"{c} (AUC {aucs[i]:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("One-vs-rest ROC curves (test)"); plt.legend(); savefig("disease_roc"); plt.show()

In [ ]:
# Evaluate the comparison models on test too - for the report table only (selection was already fixed by validation)
if RUN_ARCH_COMPARISON and len(comp_df):
    for arch in ARCHS:
        m = keras.models.load_model(DIRS["models"] / f"compare_{arch}.keras")
        mt = metrics_from_probs(y_test, predict_probs(m, test_df, "dx", DX_CLASSES, arch), DX_CLASSES)
        comp_df.loc[arch, "test_accuracy"] = mt["accuracy"]
        comp_df.loc[arch, "test_macro_f1"] = mt["macro_f1"]
        del m; keras.backend.clear_session(); gc.collect()
    comp_df.loc[BEST_ARCH + " (final, full schedule)"] = {
        "img_size": IMG_SIZES[BEST_ARCH], "params_M": round(disease_model.count_params() / 1e6, 1),
        "train_min": round(disease_hist.get("train_minutes", float("nan")), 1),
        **{f"val_{k}": v for k, v in VAL_METRICS.items()},
        "test_accuracy": TEST_METRICS["accuracy"], "test_macro_f1": TEST_METRICS["macro_f1"]}
    cols = ["img_size", "params_M", "train_min", "val_accuracy", "val_macro_f1", "val_balanced_accuracy",
            "test_accuracy", "test_macro_f1"]
    display(comp_df[cols].round(4))
    comp_df.to_csv(DIRS["reports"] / "architecture_comparison.csv")

## 14. If the 80 % target was not reached: diagnosis and systematic improvements

The cell below prints an automatic diagnosis. Then change **one** setting at a time in the configuration cell,
set `RESUME = False` (or delete the model file), retrain, and judge each change on **validation** — never on test.

Suggested order (largest expected effect first):
1. Use `EfficientNetB3` (300 px) if it was not selected, or raise `IMG_SIZES["EfficientNetB3"]` to 380.
2. Longer final fine-tuning: increase the epochs of the last stage in `FINAL_SCHEDULE` and `EARLY_STOP_PATIENCE`.
3. Lower `CLASS_WEIGHT_POWER` to 0.25 or 0 (raises accuracy, lowers minority recall — report both).
4. Try `LOSS = "focal"`; try `BATCH_SIZE = 16` with a lower stage-3 LR.
5. Ensemble the two best architectures by averaging probabilities (decide on validation).

In [ ]:
gap = VAL_METRICS["accuracy"] - TEST_METRICS["accuracy"]
print(f"Validation accuracy {VAL_METRICS['accuracy']:.3f} vs test {TEST_METRICS['accuracy']:.3f} (gap {gap:+.3f})")
if abs(gap) > 0.03:
    print(" -> sizeable val/test gap: small minority classes make both estimates noisy; consider a different seed or k-fold CV.")
tr_acc = disease_hist["accuracy"][-1]; va_acc = disease_hist["val_accuracy"][-1]
if tr_acc - va_acc > 0.10:
    print(f" -> overfitting signal (train {tr_acc:.3f} vs val {va_acc:.3f}): more augmentation/dropout or fewer unfrozen layers.")
elif tr_acc < 0.85:
    print(f" -> underfitting signal (train accuracy {tr_acc:.3f}): train longer, unfreeze more, larger backbone/resolution.")
weak = PER_CLASS_TEST.sort_values("per-class accuracy (recall)").head(3)
print("\nWeakest classes (recall):"); print(weak[["support", "per-class accuracy (recall)"]].round(3).to_string())
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], DX_CLASSES[i], DX_CLASSES[j]) for i in range(len(DX_CLASSES)) for j in range(len(DX_CLASSES))), reverse=True)[:5]
print("\nMost frequent confusions (true -> predicted):")
for n, a, b in pairs:
    print(f"  {a:>5} -> {b:<5} {n}")

## 15. Anatomical site classifier (separate model)

Site labels come from metadata. Dermoscopic close-ups usually show little body context, so this task is much harder than
lesion classification — the meaningful comparison is against the **majority-class baseline**, not against 80 %.
`unknown` is dropped and sites with fewer than `LOC_MIN_IMAGES` images are excluded. The **same lesion-level split** is reused,
so there is no leakage between tasks either.

In [ ]:
LOC_ARCH_USED = LOC_ARCH or BEST_ARCH
loc_counts_train = train_df["localization"].value_counts()
LOC_CLASSES = sorted([c for c, n in df["localization"].value_counts().items()
                      if c != "unknown" and n >= LOC_MIN_IMAGES])
excluded = sorted(set(df["localization"].unique()) - set(LOC_CLASSES))
print("Site classes:", LOC_CLASSES); print("Excluded:", excluded)

loc_train = train_df[train_df["localization"].isin(LOC_CLASSES)].reset_index(drop=True)
loc_val = val_df[val_df["localization"].isin(LOC_CLASSES)].reset_index(drop=True)
loc_test = test_df[test_df["localization"].isin(LOC_CLASSES)].reset_index(drop=True)
LOC_WEIGHTS = class_weights(loc_train["localization"], LOC_CLASSES)
print({k: len(v) for k, v in [("train", loc_train), ("val", loc_val), ("test", loc_test)]})

loc_model, loc_hist = train_model(LOC_ARCH_USED, LOC_CLASSES, loc_train, loc_val, "localization", LOC_WEIGHTS,
                                  LOC_SCHEDULE, run_name=f"location_{LOC_ARCH_USED}")
plot_history(loc_hist, f"Site model ({LOC_ARCH_USED})", "location_training_curves")

In [ ]:
y_lt = y_index(loc_test, "localization", LOC_CLASSES)
loc_probs = predict_probs(loc_model, loc_test, "localization", LOC_CLASSES, LOC_ARCH_USED)
LOC_TEST_METRICS = metrics_from_probs(y_lt, loc_probs, LOC_CLASSES)
majority = loc_train["localization"].value_counts().idxmax()
LOC_TEST_METRICS["majority_baseline_accuracy"] = float((loc_test["localization"] == majority).mean())
display(pd.Series(LOC_TEST_METRICS).round(4).to_frame("site model — TEST"))

print(classification_report(y_lt, loc_probs.argmax(1), labels=range(len(LOC_CLASSES)),
                            target_names=LOC_CLASSES, digits=3, zero_division=0))
lcm = confusion_matrix(y_lt, loc_probs.argmax(1), labels=range(len(LOC_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(lcm / np.maximum(lcm.sum(1, keepdims=True), 1), annot=True, fmt=".2f", cmap="Greens",
            xticklabels=LOC_CLASSES, yticklabels=LOC_CLASSES)
plt.title("Site model — row-normalised confusion matrix (test)"); plt.xlabel("Predicted"); plt.ylabel("True")
savefig("location_confusion_matrix"); plt.show()

## 16. Explainable AI — Grad-CAM

Grad-CAM weights the last convolutional feature maps by the gradient of the predicted class **logit**, showing which image
regions most influenced the prediction. It is a visual sanity check (is the model looking at the lesion, or at rulers,
hair, ink marks, vignetting?) — **not** evidence that a prediction is medically correct.

In [ ]:
def find_layer(model, name):
    try:
        return model.get_layer(name)
    except ValueError:
        for l in model.layers:
            if isinstance(l, keras.Model):
                try: return l.get_layer(name)
                except ValueError: pass
    # fallback: last layer with a 4-D output
    return [l for l in model.layers if len(getattr(l, "output", np.zeros(0)).shape) == 4][-1]

def gradcam_heatmap(model, x, layer_name, class_idx=None):
    conv_layer = find_layer(model, layer_name)
    grad_model = keras.Model(model.inputs, [conv_layer.output, model.get_layer("logits").output])
    x = tf.convert_to_tensor(x)
    with tf.GradientTape() as tape:
        conv_out, logits = grad_model(x, training=False)
        conv_out = tf.cast(conv_out, tf.float32)
        if class_idx is None:
            class_idx = int(tf.argmax(logits[0]))
        score = tf.cast(logits[:, class_idx], tf.float32)
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1)).numpy()
    return cam / (cam.max() + 1e-8), class_idx

def overlay(img_uint8, cam, alpha=0.4):
    cam_img = np.array(Image.fromarray(np.uint8(255 * cam)).resize(img_uint8.shape[1::-1], Image.BILINEAR)) / 255.0
    heat = plt.cm.jet(cam_img)[..., :3] * 255
    return np.uint8((1 - alpha) * img_uint8 + alpha * heat)

def load_image_array(path, size):
    with Image.open(path) as im:
        return np.array(im.convert("RGB").resize((size, size), Image.BILINEAR))

# Show correct and incorrect test predictions
size = IMG_SIZES[BEST_ARCH]; prep = preprocess_for(BEST_ARCH)
correct = np.where(test_pred == y_test)[0]; wrong = np.where(test_pred != y_test)[0]
rng = np.random.default_rng(SEED)
picks = list(rng.choice(correct, min(4, len(correct)), replace=False)) + list(rng.choice(wrong, min(4, len(wrong)), replace=False))
fig, axes = plt.subplots(len(picks), 2, figsize=(7, 3.3 * len(picks)))
axes = np.atleast_2d(axes)
for r, i in enumerate(picks):
    img = load_image_array(test_df.loc[i, "path"], size)
    cam, ci = gradcam_heatmap(disease_model, prep(img[None].astype("float32")), GRADCAM_LAYERS[BEST_ARCH])
    axes[r, 0].imshow(img); axes[r, 0].set_title(f"true: {DX_CLASSES[y_test[i]]}", fontsize=9)
    axes[r, 1].imshow(overlay(img, cam))
    axes[r, 1].set_title(f"pred: {DX_CLASSES[ci]} ({test_probs[i, ci]:.0%}) {'✔' if ci == y_test[i] else '✘'}", fontsize=9)
    for a in axes[r]: a.axis("off")
plt.tight_layout(); savefig("gradcam_examples"); plt.show()

## 17. Save models, configuration and make them downloadable

In [ ]:
MODEL_CONFIG = {
    "disease": {"file": f"disease_{BEST_ARCH}.keras", "arch": BEST_ARCH, "img_size": IMG_SIZES[BEST_ARCH],
                "classes": DX_CLASSES, "class_names": DX_NAMES, "gradcam_layer": GRADCAM_LAYERS[BEST_ARCH],
                "use_tta": bool(USE_TTA)},
    "location": {"file": f"location_{LOC_ARCH_USED}.keras", "arch": LOC_ARCH_USED,
                 "img_size": IMG_SIZES[LOC_ARCH_USED], "classes": LOC_CLASSES},
    "preprocessing": "float32 RGB in [0,255], resized (bilinear) to img_size; EfficientNet: none, "
                     "MobileNetV2: x/127.5-1, ResNet50: RGB->BGR minus ImageNet mean",
    "test_accuracy": float(TEST_METRICS["accuracy"]), "seed": SEED,
}
(DIRS["models"] / "model_config.json").write_text(json.dumps(MODEL_CONFIG, indent=2))

bundle = PROJECT_DIR / "skin_lesion_models.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [MODEL_CONFIG["disease"]["file"], MODEL_CONFIG["location"]["file"], "model_config.json"]:
        z.write(DIRS["models"] / f, arcname=f)
print("Saved:", bundle, f"({bundle.stat().st_size/1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))

### Reuse saved models in a new session
Run sections 1–2 (setup + Drive), then this cell. It is self-contained and does not need the dataset.

In [ ]:
def load_saved_models(models_dir=DIRS["models"]):
    cfg = json.loads((Path(models_dir) / "model_config.json").read_text())
    dm = keras.models.load_model(Path(models_dir) / cfg["disease"]["file"])
    lm = keras.models.load_model(Path(models_dir) / cfg["location"]["file"])
    return dm, lm, cfg

disease_model, loc_model, MODEL_CONFIG = load_saved_models()
print("Loaded:", MODEL_CONFIG["disease"]["file"], "+", MODEL_CONFIG["location"]["file"])

## 18. Prediction interface — upload a skin image

Upload a **dermatoscopic** image (like those in HAM10000). Phone photos, non-lesion images or other skin conditions are outside what
the model was trained on; it will still output one of the 7 classes, so treat such results as meaningless.

In [ ]:
def _prep_input(img_uint8, arch):
    return preprocess_for(arch)(tf.convert_to_tensor(img_uint8[None].astype("float32")))

def _predict(model, x, tta):
    views = [x]
    if tta:
        views += [tf.image.flip_left_right(x), tf.image.flip_up_down(x), tf.image.rot90(x, k=1)]
    return np.mean([model(v, training=False).numpy()[0] for v in views], axis=0)

def analyze_image(path, cfg=None):
    cfg = cfg or MODEL_CONFIG
    dc, lc = cfg["disease"], cfg["location"]
    try:
        img_d = load_image_array(path, dc["img_size"])
        img_l = load_image_array(path, lc["img_size"])
    except Exception as e:
        print(f"Could not read '{path}': {e}"); return None

    xd = _prep_input(img_d, dc["arch"])
    p_dx = _predict(disease_model, xd, dc["use_tta"])
    p_loc = _predict(loc_model, _prep_input(img_l, lc["arch"]), False)
    top3 = np.argsort(p_dx)[::-1][:3]
    cam, _ = gradcam_heatmap(disease_model, xd, dc["gradcam_layer"], class_idx=int(top3[0]))
    loc_i = int(p_loc.argmax())

    res = {"disease": dc["classes"][top3[0]], "disease_name": dc["class_names"][dc["classes"][top3[0]]],
           "disease_confidence": float(p_dx[top3[0]]),
           "top3": [(dc["classes"][i], dc["class_names"][dc["classes"][i]], float(p_dx[i])) for i in top3],
           "location": lc["classes"][loc_i], "location_confidence": float(p_loc[loc_i])}

    fig = plt.figure(figsize=(14, 6.4))
    gs = fig.add_gridspec(2, 3, height_ratios=[4, 1.6], width_ratios=[1, 1, 1.3])
    a0, a1, a2 = [fig.add_subplot(gs[0, i]) for i in range(3)]
    a3 = fig.add_subplot(gs[1, :])
    a0.imshow(img_d); a0.set_title("Uploaded image"); a0.axis("off")
    a1.imshow(overlay(img_d, cam)); a1.set_title(f"Grad-CAM for '{res['disease']}'"); a1.axis("off")
    names = [f"{c}\n{n[:24]}" for c, n, _ in res["top3"]][::-1]
    a2.barh(names, [p for *_, p in res["top3"]][::-1], color=["#9ecae1", "#6baed6", "#2171b5"])
    a2.set_xlim(0, 1); a2.set_title("Top-3 lesion classes (model probability)")
    for k, (_, _, p) in enumerate(res["top3"][::-1]):
        a2.text(min(p + 0.02, 0.85), k, f"{p:.1%}", va="center")
    a3.axis("off")
    low = res["disease_confidence"] < LOW_CONFIDENCE_THRESHOLD
    txt = (f"Predicted lesion class: {res['disease_name']} ({res['disease']})  —  "
           f"model confidence {res['disease_confidence']:.1%}{'  ⚠ LOW' if low else ''}\n"
           f"Predicted body site:    {res['location']}  —  model confidence {res['location_confidence']:.1%}")
    note = ("Model confidence is the softmax score over the classes the model was trained on. It is NOT the probability "
            "of having this condition\nand NOT a diagnosis. Research prototype only — see a dermatologist for any concerning lesion.")
    a3.text(0, 0.75, txt, va="center", fontsize=11, family="monospace", weight="bold")
    a3.text(0, 0.15, note, va="center", fontsize=9, color="dimgray")
    plt.tight_layout(); plt.show()

    print(f"Disease: {res['disease_name']} ({res['disease']}) — model confidence {res['disease_confidence']:.1%}")
    print("Top-3:", ", ".join(f"{c} {p:.1%}" for c, _, p in res["top3"]))
    print(f"Body site: {res['location']} — model confidence {res['location_confidence']:.1%}")
    if low:
        print("⚠ Low model confidence: the image may be ambiguous or unlike the training data.")
    return res

def upload_and_analyze():
    if not IN_COLAB:
        print("Use analyze_image('path/to/image.jpg') outside Colab."); return
    from google.colab import files
    os.makedirs("/content/uploads", exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join("/content/uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        print(f"\n===== {name} =====")
        analyze_image(p)

# Demo on one held-out test image (ground truth shown for reference)
demo = test_df.sample(1, random_state=7).iloc[0]
print(f"Demo test image — ground truth: {demo['dx']} / {demo['localization']}")
_ = analyze_image(demo["path"])

In [ ]:
# ▶ Run this cell and choose one or more images to analyse
upload_and_analyze()

## 19. Final performance report (saved to Drive as Markdown)

In [ ]:
def fmt(x):
    return f"{x:.4f}" if isinstance(x, (float, np.floating)) and not np.isnan(x) else str(x)

lines = [
    "# Performance report — HAM10000 skin lesion & body-site classification", "",
    f"Generated {time.strftime('%Y-%m-%d %H:%M')} · seed {SEED} · TensorFlow {tf.__version__}", "",
    "## Data",
    f"- Images after cleaning: {len(df)} ({df['lesion_id'].nunique()} lesions); corrupted removed: {len(bad)}; "
    f"byte-identical duplicates removed: {len(dup) - dup['md5'].nunique() if len(dup) else 0}",
    f"- Lesion-level stratified split: train {len(train_df)}, val {len(val_df)}, test {len(test_df)} images; "
    "no lesion/image/file overlap (asserted)",
    f"- Class weighting power: {CLASS_WEIGHT_POWER} · loss: {LOSS} (label smoothing {LABEL_SMOOTHING})", "",
    "## Architecture comparison (selection by validation accuracy)",
]
if len(comp_df):
    cols = [c for c in ["img_size", "params_M", "train_min", "val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1"] if c in comp_df]
    lines += ["| model | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
    for name, row in comp_df[cols].iterrows():
        lines.append(f"| {name} | " + " | ".join(fmt(row[c]) for c in cols) + " |")
    lines.append("\nComparison models used the short schedule; test columns were added after selection, for reference only.")
else:
    lines.append(f"Comparison skipped; used {BEST_ARCH}.")
target_met = TEST_METRICS["accuracy"] >= TARGET_ACC
lines += ["", f"## Final disease model: {BEST_ARCH} (TTA: {USE_TTA})", "",
          "| metric | validation | test (all images) | test (1 image/lesion) |", "|---|---|---|---|"]
for k in TEST_METRICS:
    lines.append(f"| {k} | {fmt(VAL_METRICS.get(k, np.nan))} | {fmt(TEST_METRICS[k])} | {fmt(TEST_METRICS_1PL.get(k, np.nan))} |")
lines += ["", f"**80 % test-accuracy target: {'MET' if target_met else 'NOT MET'}** "
          f"(test accuracy {TEST_METRICS['accuracy']:.2%}; majority-class baseline {(y_test == DX_CLASSES.index('nv')).mean():.2%})",
          "", "### Per-class test performance", "",
          "| class | support | recall (per-class acc.) | precision | F1 | ROC-AUC |", "|---|---|---|---|---|---|"]
for c, r in PER_CLASS_TEST.iterrows():
    lines.append(f"| {c} ({DX_NAMES[c]}) | {int(r['support'])} | {r['per-class accuracy (recall)']:.3f} | "
                 f"{r['precision']:.3f} | {r['f1']:.3f} | {fmt(r['ROC-AUC (one-vs-rest)'])} |")
lines += ["", f"## Body-site model: {LOC_ARCH_USED} ({len(LOC_CLASSES)} classes)", "",
          "| metric | test |", "|---|---|"] + [f"| {k} | {fmt(v)} |" for k, v in LOC_TEST_METRICS.items()]
lines += ["", "## Limitations",
          "- Research prototype; not clinically validated. Classifies only the 7 HAM10000 categories.",
          "- HAM10000 is dermoscopic, mostly from two centres (Vienna, Queensland) and predominantly light skin types; "
          "performance on other populations, devices or clinical photos is unknown.",
          "- Strong class imbalance (nv ≈ 67 %): accuracy must be read with macro-F1 and per-class recall.",
          "- Model confidence is a softmax score, not a calibrated probability of disease.",
          "- Body-site labels come from metadata and are weakly visible in close-ups; the site model is a secondary, low-reliability output.",
          "- Grad-CAM shows influential regions, not medical correctness."]
report = "\n".join(lines)
(DIRS["reports"] / "performance_report.md").write_text(report)
from IPython.display import Markdown
display(Markdown(report))
print("Saved:", DIRS["reports"] / "performance_report.md")

## 20. Research and medical limitations

**What the system is.** An educational prototype that maps a dermatoscopic image to one of seven HAM10000 categories and,
separately, guesses the body site recorded in metadata. It cannot recognise any other skin disease, cannot say "this is not a
lesion", and has never been validated prospectively or on patients.

**HAM10000 limitations.**
* Dermoscopy images from a small number of sources (mainly Vienna and Queensland) with particular devices and protocols.
* About half of the labels are histopathology-confirmed; the rest come from follow-up, expert consensus or confocal microscopy (`dx_type`), so some label noise exists.
* Several images per lesion: evaluation must be lesion-grouped (done here), otherwise accuracy is inflated.

**Class imbalance.** `nv` is ~67 % of images while `df` and `vasc` are ~1 % each. A trivial model reaches ~67 % accuracy;
macro-F1, balanced accuracy and per-class recall (especially for melanoma) are the more informative numbers. Weighting
trades overall accuracy for minority-class recall.

**Demographic and acquisition bias.** Patients are predominantly fair-skinned (Fitzpatrick I–III); skin tone is not
recorded, so subgroup performance cannot be measured. Artefacts such as rulers, ink markings, hair, gel bubbles and
vignetting can correlate with classes and be learned as shortcuts — Grad-CAM can help spot this.

**Lesion classification vs. anatomical localisation.** Diagnosis is a property of the lesion's appearance; the body site is a property of
*where the photo was taken*, which a 600×450 close-up mostly does not show. The site model can only exploit indirect cues
(skin texture, hair density, sun damage, lesion-type priors per site), so its accuracy is far lower and should be judged against the
majority baseline. The two models are independent; neither uses the other's prediction.

**Why confidence ≠ diagnostic certainty.** The softmax output is a relative score across seven classes the model was forced to
choose between. Deep networks are often over-confident, scores are not calibrated to real-world prevalence, and an image from
outside the training distribution (another disease, a phone photo, a non-skin image) can receive a very high score. A high score is
not a probability that the person has the disease.